# XGBoost

Part of the reference implementation for *Reliability-Aware Model-Dependent Uncertainty Placement under Data Asymmetry for Probabilistic Load Forecasting*. Both uncertainty placements are produced: the modular placement (post-hoc contextual residual quantiles, route 1) and the integrated placement (in-model quantiles, route 2).

Run the cells in order. Outputs go to `outputs/`. 

## 1. Symmetric protocol ($X = X'$)
Training and testing at 5-minute resolution with all channels. Writes `route1` (point + modular quantiles) and `route2` (integrated quantiles) CSVs for Dec 24 and Dec 30, 2024.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# XGBoost symmetric evaluation at 5-minute resolution
# FEATURE-ALIGNED + STANDARDIZED ROUTE1 VERSION
#
# Train: V1 only
# Validate: V2 excluding Dec 24 and Dec 30
# Test: Dec 24 and Dec 30 from V2
#
# Route1:
#   - point XGBoost trained ONCE on V1 only
#   - forward one-step prediction on validation
#   - NO retraining inside validation
#   - residual buckets built from validation residuals
#
# Route2:
#   - integrated probabilistic XGBoost via quantile models
# ============================


import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import norm

warnings.filterwarnings("ignore")

# =========================
# 0) Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================
# 1) Paths
# =========================
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/xgb_out_symmetric_5min_route1_standardized"
os.makedirs(OUT_DIR, exist_ok=True)

MERGED_ROUTE1 = os.path.join(OUT_DIR, "merged_route1_point_uq_series_xgb_symmetric_5min.csv")
MERGED_ROUTE2 = os.path.join(OUT_DIR, "merged_route2_predictions_prob_xgb_symmetric_5min.csv")

# =========================
# 2) Timestamp rules
# =========================
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# =========================
# 3) Forecast settings
# =========================
STEPS_PER_HOUR = 12
HORIZON_HOURS = 24
PRED_LEN = HORIZON_HOURS * STEPS_PER_HOUR   # 288

Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

FEATURE_COLS = [
    "temp_c",
    "voltage_v",
    "amperage_a",
    "generation_w",
    "hour",
    "dayofweek",
    "minute_of_day",
    "slot_5min",
    "lag_5min",
    "lag_10min",
    "lag_1h",
    "lag_24h",
    "ma_1h",
    "ma_3h",
]

VAL_STRIDE = 1   # set >1 only if you later want faster but sparser Route1 residuals

# =========================
# 4) Helpers
# =========================
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found {df.columns.tolist()}")

    print(
        f"[{os.path.basename(path)}] "
        f"power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'"
    )

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    out["hour"] = out["timestamp"].dt.hour
    out["dayofweek"] = out["timestamp"].dt.dayofweek
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["slot_5min"] = out["minute_of_day"] // 5

    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"] = df["minute_of_day"] // 5

    return df

def add_engineered_features_5min(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["lag_5min"] = df["power_w"].shift(1)
    df["lag_10min"] = df["power_w"].shift(2)
    df["lag_1h"] = df["power_w"].shift(12)
    df["lag_24h"] = df["power_w"].shift(288)

    df["ma_1h"] = df["power_w"].shift(1).rolling(window=12, min_periods=1).mean()
    df["ma_3h"] = df["power_w"].shift(1).rolling(window=36, min_periods=1).mean()

    df = df.bfill().ffill()
    return df

def build_tabular_supervised_5min(df):
    """
    One-step supervised mapping:
        X_t -> y_{t+1}
    where one step = 5 minutes
    """
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["y_next"] = df["power_w"].shift(-1)
    out = df.dropna(subset=FEATURE_COLS + ["y_next"]).reset_index(drop=True)

    X = out[FEATURE_COLS].to_numpy(dtype=float)
    y = out["y_next"].to_numpy(dtype=float)

    return X, y, out

def make_feature_row_5min(temp_c, voltage_v, amperage_a, generation_w, ts, history_power):
    lag_5min = history_power[-1]
    lag_10min = history_power[-2] if len(history_power) >= 2 else history_power[-1]
    lag_1h = history_power[-12] if len(history_power) >= 12 else history_power[0]
    lag_24h = history_power[-288] if len(history_power) >= 288 else history_power[0]

    ma_1h = float(np.mean(history_power[-12:])) if len(history_power) >= 12 else float(np.mean(history_power))
    ma_3h = float(np.mean(history_power[-36:])) if len(history_power) >= 36 else float(np.mean(history_power))

    minute_of_day = ts.hour * 60 + ts.minute
    slot_5min = minute_of_day // 5

    return np.array([[
        float(temp_c),
        float(voltage_v),
        float(amperage_a),
        float(generation_w),
        int(ts.hour),
        int(ts.dayofweek),
        int(minute_of_day),
        int(slot_5min),
        float(lag_5min),
        float(lag_10min),
        float(lag_1h),
        float(lag_24h),
        float(ma_1h),
        float(ma_3h),
    ]], dtype=float)

def train_xgb_point(Xtr, ytr, Xva, yva):
    dtrain = xgb.DMatrix(Xtr, label=ytr, feature_names=FEATURE_COLS)
    dval = xgb.DMatrix(Xva, label=yva, feature_names=FEATURE_COLS)

    params = dict(
        objective="reg:squarederror",
        eta=0.03,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        **{"lambda": 1.0},
        min_child_weight=1.0,
        tree_method="hist",
        seed=SEED,
    )

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=6000,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=120,
        verbose_eval=False
    )
    return booster, dtrain, dval

def _xgb_supports_quantile_native(dtrain, dval):
    try:
        tmp_params = {
            "objective": "reg:quantileerror",
            "quantile_alpha": 0.5,
            "eta": 0.1,
            "max_depth": 2,
            "tree_method": "hist",
            "seed": SEED,
        }
        _ = xgb.train(
            params=tmp_params,
            dtrain=dtrain,
            num_boost_round=5,
            evals=[(dval, "val")],
            verbose_eval=False
        )
        return True
    except Exception as e:
        print("Quantile objective check failed:", repr(e))
        return False

def train_xgb_quantile(Xtr, ytr, Xva, yva, alpha):
    dtrain = xgb.DMatrix(Xtr, label=ytr, feature_names=FEATURE_COLS)
    dval = xgb.DMatrix(Xva, label=yva, feature_names=FEATURE_COLS)

    params = dict(
        objective="reg:quantileerror",
        quantile_alpha=float(alpha),
        eta=0.03,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        **{"lambda": 1.0},
        min_child_weight=1.0,
        tree_method="hist",
        seed=SEED,
    )

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=7000,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=100,
        verbose_eval=False
    )
    return booster

def forward_one_step_validation_predictions(model, train_df, val_df, stride=1):
    """
    Standardized Route1:
    - use the main point model trained on V1 only
    - run forward one-step prediction over validation
    - update history with ACTUAL validation observation
    - no retraining inside validation
    """
    hist_df = train_df.copy().sort_values("timestamp").reset_index(drop=True)
    val_df = val_df.copy().sort_values("timestamp").reset_index(drop=True)

    if len(hist_df) < 288:
        raise ValueError("Need at least 288 historical points before validation for lag_24h.")

    history_power = hist_df["power_w"].to_numpy(dtype=float).tolist()

    preds = []
    actuals = []
    ts_list = []

    for j in range(len(val_df)):
        if j % 1000 == 0:
            print(f"  validation rolling progress: {j}/{len(val_df)}")

        row = val_df.iloc[j]
        ts = pd.Timestamp(row["timestamp"])

        feat = make_feature_row_5min(
            temp_c=row["temp_c"],
            voltage_v=row["voltage_v"],
            amperage_a=row["amperage_a"],
            generation_w=row["generation_w"],
            ts=ts,
            history_power=history_power
        )

        dm = xgb.DMatrix(feat, feature_names=FEATURE_COLS)
        yhat = float(model.predict(dm)[0])
        ytrue = float(row["power_w"])

        if (j % stride) == 0:
            preds.append(yhat)
            actuals.append(ytrue)
            ts_list.append(ts)

        # Update with ACTUAL validation observation
        history_power.append(ytrue)

    out = pd.DataFrame({
        "timestamp": ts_list,
        "actual": actuals,
        "pred": preds
    })
    return out

def build_route1_residual_buckets_from_val(model, train_df, val_df):
    df_cal = forward_one_step_validation_predictions(model, train_df, val_df, stride=VAL_STRIDE)

    if df_cal.empty:
        raise ValueError("Validation predictions are empty.")

    df_cal["residual"] = df_cal["actual"] - df_cal["pred"]

    clip_val = np.percentile(np.abs(df_cal["residual"].to_numpy()), 99)
    df_cal["residual"] = np.clip(df_cal["residual"], -clip_val, clip_val)

    df_cal["slot_5min"] = df_cal["timestamp"].dt.hour * 12 + (df_cal["timestamp"].dt.minute // 5)
    df_cal["dayofweek"] = df_cal["timestamp"].dt.dayofweek

    res_q = (
        df_cal.groupby(["slot_5min", "dayofweek"])["residual"]
              .quantile([Q_LO, Q_MID, Q_HI])
              .unstack()
              .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
              .reset_index()
    )

    sigma_global = float(np.std(df_cal["residual"].to_numpy()))
    sigma_global = max(sigma_global, 1e-6)

    return res_q, sigma_global, df_cal

def apply_route1_buckets_5min(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["slot_5min"] = tmp["timestamp"].dt.hour * 12 + (tmp["timestamp"].dt.minute // 5)
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek

    tmp = tmp.merge(res_q, on=["slot_5min", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

# =========================
# 5) Load V1 and V2
# =========================
print("Loading V1 and V2 at 5-minute resolution...")
v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)
full_5m = add_engineered_features_5min(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

print("✅ V1 5-min samples:", len(v1_5m))
print("✅ V2 5-min samples:", len(v2_5m))

# =========================
# 6) Symmetric split
# =========================
test_mask = v2_5m["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df = v1_5m.copy().reset_index(drop=True)
val_df = v2_5m[~test_mask].copy().reset_index(drop=True)
test_df_all = v2_5m[test_mask].copy().reset_index(drop=True)

print("✅ Train samples:", len(train_df))
print("✅ Val samples  :", len(val_df))
print("✅ Test samples :", len(test_df_all))

for d in TEST_DAYS:
    cur = test_df_all[test_df_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != PRED_LEN:
        raise ValueError(f"Test day {d.date()} has {len(cur)} points, expected {PRED_LEN}.")

# =========================
# Skip rerun if outputs exist
# =========================
if os.path.exists(MERGED_ROUTE1) and os.path.exists(MERGED_ROUTE2):
    print("\n✅ Final merged CSVs already exist. Skipping full pipeline.")
    print("Route1:", MERGED_ROUTE1)
    print("Route2:", MERGED_ROUTE2)

else:
    # =========================
    # 7) Train point model on V1 only
    # =========================
    print("\nTraining Route1 point model on V1 only...")
    Xtr_all, ytr_all, df_train_sup = build_tabular_supervised_5min(train_df)

    split_train = int(0.9 * len(df_train_sup))
    Xtr, ytr = Xtr_all[:split_train], ytr_all[:split_train]
    Xva, yva = Xtr_all[split_train:], ytr_all[split_train:]

    t0 = time.time()
    xgb_point, dtrain, dval = train_xgb_point(Xtr, ytr, Xva, yva)
    print(f"Point model fit time: {time.time()-t0:.2f}s | train_samples={len(ytr_all)} | best_iter={xgb_point.best_iteration}")

    # =========================
    # 8) Build Route1 residual buckets from standardized forward validation
    # =========================
    print("\nBuilding Route1 residual buckets from validation using forward one-step prediction...")
    res_q, sigma_global, df_res = build_route1_residual_buckets_from_val(xgb_point, train_df, val_df)

    resq_path = os.path.join(OUT_DIR, "xgb_route1_residual_quantiles_from_v2val_5min.csv")
    resid_path = os.path.join(OUT_DIR, "xgb_route1_residuals_from_v2val_5min.csv")
    res_q.to_csv(resq_path, index=False)
    df_res.to_csv(resid_path, index=False)

    print("✅ Saved residual buckets:", resq_path)
    print("✅ Saved residual table  :", resid_path)
    print(f"Fallback sigma_global: {sigma_global:.4f}")

    # =========================
    # 9) Train Route2 quantile models on V1 only
    # =========================
    print("\nTraining Route2 probabilistic models on V1 only...")
    HAS_Q = _xgb_supports_quantile_native(dtrain, dval)

    if HAS_Q:
        t0 = time.time()
        xgb_q10 = train_xgb_quantile(Xtr, ytr, Xva, yva, Q_LO)
        xgb_q50 = train_xgb_quantile(Xtr, ytr, Xva, yva, Q_MID)
        xgb_q90 = train_xgb_quantile(Xtr, ytr, Xva, yva, Q_HI)
        print(f"Quantile models trained in {time.time()-t0:.2f}s")
    else:
        print("⚠️ Native quantile objective unavailable. Route2 will use Normal fallback around point predictions.")

    # =========================
    # 10) Test on Dec 24 and Dec 30
    # =========================
    print("\nTesting on Dec 24 and Dec 30...")

    r1_uq_day_files = []
    r2_prob_day_files = []

    for k, day_start in enumerate(TEST_DAYS, start=1):
        print("\n" + "=" * 70)
        print(f"Test day {k}: {day_start.date()}")

        day_end = day_start + pd.Timedelta(days=1)

        test_df = test_df_all[
            (test_df_all["timestamp"] >= day_start) &
            (test_df_all["timestamp"] < day_end)
        ].copy().reset_index(drop=True)

        if len(test_df) != PRED_LEN:
            raise ValueError(f"Test day {day_start.date()} does not have {PRED_LEN} samples.")

        y_true = test_df["power_w"].to_numpy(dtype=float)

        hist_df = full_5m[full_5m["timestamp"] <= (day_start - pd.Timedelta(minutes=5))].copy().reset_index(drop=True)
        if len(hist_df) < 288:
            raise ValueError(f"Not enough history before {day_start.date()}.")

        history_power = hist_df["power_w"].to_numpy(dtype=float).tolist()

        point_pred = np.zeros(PRED_LEN, dtype=float)
        q10_prob = np.zeros(PRED_LEN, dtype=float)
        q50_prob = np.zeros(PRED_LEN, dtype=float)
        q90_prob = np.zeros(PRED_LEN, dtype=float)

        for h in range(PRED_LEN):
            row = test_df.iloc[h]
            ts = pd.Timestamp(row["timestamp"])

            feat = make_feature_row_5min(
                temp_c=row["temp_c"],
                voltage_v=row["voltage_v"],
                amperage_a=row["amperage_a"],
                generation_w=row["generation_w"],
                ts=ts,
                history_power=history_power
            )

            dm = xgb.DMatrix(feat, feature_names=FEATURE_COLS)

            yhat = float(xgb_point.predict(dm)[0])
            point_pred[h] = yhat

            if HAS_Q:
                a = float(xgb_q10.predict(dm)[0])
                b = float(xgb_q50.predict(dm)[0])
                c = float(xgb_q90.predict(dm)[0])
                aa, bb, cc = enforce_non_crossing(np.array([a]), np.array([b]), np.array([c]))
                q10_prob[h], q50_prob[h], q90_prob[h] = aa[0], bb[0], cc[0]
            else:
                q50_prob[h] = yhat
                q10_prob[h] = yhat + norm.ppf(0.10) * sigma_global
                q90_prob[h] = yhat + norm.ppf(0.90) * sigma_global
                aa, bb, cc = enforce_non_crossing(
                    np.array([q10_prob[h]]),
                    np.array([q50_prob[h]]),
                    np.array([q90_prob[h]])
                )
                q10_prob[h], q50_prob[h], q90_prob[h] = aa[0], bb[0], cc[0]

            # recursive update with predicted power for test rollout
            history_power.append(yhat)

        # Route1
        p10_uq, p50_uq, p90_uq = apply_route1_buckets_5min(
            test_df["timestamp"].values, point_pred, res_q
        )

        out_r1 = pd.DataFrame({
            "timestamp": test_df["timestamp"].values,
            "actual": y_true,
            "point_pred": point_pred,
            "q10": p10_uq,
            "q50": p50_uq,
            "q90": p90_uq
        })
        p_r1 = os.path.join(OUT_DIR, f"xgb_route1_uq_symmetric_5min_day{k}.csv")
        out_r1.to_csv(p_r1, index=False)
        r1_uq_day_files.append(p_r1)

        # Route2
        out_r2 = pd.DataFrame({
            "timestamp": test_df["timestamp"].values,
            "actual": y_true,
            "q10": q10_prob,
            "q50": q50_prob,
            "q90": q90_prob
        })
        p_r2 = os.path.join(OUT_DIR, f"xgb_route2_prob_symmetric_5min_day{k}.csv")
        out_r2.to_csv(p_r2, index=False)
        r2_prob_day_files.append(p_r2)

        print("✅ Saved:")
        print("  Route1_UQ   :", p_r1)
        print("  Route2_Prob :", p_r2)

    # =========================
    # 11) Merge outputs
    # =========================
    print("\nMerging outputs...")

    merged_route1_df = merge_daily_route1(r1_uq_day_files, MERGED_ROUTE1)
    merged_route2_df = merge_daily_route2(r2_prob_day_files, MERGED_ROUTE2)

    print("✅ Merged files written:")
    print("  Route1:", MERGED_ROUTE1)
    print("  Route2:", MERGED_ROUTE2)
    print("  OUT_DIR:", OUT_DIR)

    print("\nRoute1 head:")
    print(merged_route1_df.head())

    print("\nRoute2 head:")
    print(merged_route2_df.head())

## 2. Asymmetric protocol ($X \neq X'$)
Training on the full multivariate history; deployment on hourly data with temperature and calendar features only. Voltage, current and PV generation are filled with hour-of-day means from $D_1$.

In [ ]:
# ---- repository paths -------------------------------------------------
# REPO defaults to the parent of notebooks/ when run locally, or to
# MyDrive/uncertainty-placement on Colab. Override with SEGAN_REPO.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _default_repo = "/content/drive/MyDrive/uncertainty-placement"
except ImportError:
    _default_repo = os.path.abspath(os.path.join(os.getcwd(), ".."))
REPO     = os.environ.get("SEGAN_REPO", _default_repo)
DATA_DIR = os.environ.get("SEGAN_DATA", os.path.join(REPO, "data"))
OUT_ROOT = os.environ.get("SEGAN_OUT",  os.path.join(REPO, "outputs"))
os.makedirs(OUT_ROOT, exist_ok=True)
# -------------------------------------------------------------------------
# ============================
# XGBoost ASYMMETRIC evaluation
#
# Train: V1 only, 5-minute, multivariate
# Validate: V2 excluding Dec 24 and Dec 30
# Test: Dec 24 and Dec 30 from V2, converted to HOURLY
#
# Asymmetry:
#   - Train on rich 5-minute multivariate features
#   - Deploy on hourly temperature + time only
#
# Route1: point XGB + post-hoc residual buckets
#         residual buckets are built from V2 validation
#         BUT under deployment-style hourly degraded inference
#
# Route2: integrated probabilistic XGB via quantile models
#         trained on V1 rich regime, inferred on degraded hourly regime
# ============================

import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import norm

warnings.filterwarnings("ignore")

# =========================
# 0) Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================
# 1) Paths
# =========================
V1_PATH = os.path.join(DATA_DIR, "data_v1.0.csv")
V2_PATH = os.path.join(DATA_DIR, "data_v2.0.csv")

OUT_DIR = OUT_ROOT + "/xgb_out_asymmetric_train5min_testhourly"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# 2) Timestamp rules
# =========================
V1_START_DATE = "2023-12-01 00:00:00"
V2_START_DATE = "2024-12-01 00:00:00"
SRC_FREQ = "5min"

# =========================
# 3) Forecast settings
# =========================
Q_LO, Q_MID, Q_HI = 0.10, 0.50, 0.90

TEST_DAYS = [
    pd.Timestamp("2024-12-24"),
    pd.Timestamp("2024-12-30"),
]

EXPECTED_HOURLY_POINTS_PER_DAY = 24

# training features: 5-minute multivariate rich regime
TRAIN_FEATURES = [
    "temp_c",
    "voltage_v",
    "amperage_a",
    "generation_w",
    "hour",
    "dayofweek",
    "minute_of_day",
    "slot_5min",
    "lag_5min",
    "lag_10min",
    "lag_1h",
    "lag_24h",
    "ma_1h",
    "ma_3h",
]

# deployment / degraded known features
DEPLOY_KNOWN = [
    "temp_c",
    "hour",
    "dayofweek",
]

# =========================
# 4) Helpers
# =========================
def _find_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def enforce_non_crossing(q10, q50, q90):
    qs = np.sort(np.vstack([q10, q50, q90]).T, axis=1)
    return qs[:, 0], qs[:, 1], qs[:, 2]

def merge_daily_route1(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)
    merged.to_csv(out_path, index=False)
    return merged

def merge_daily_route2(day_files, out_path):
    parts = []
    for p in day_files:
        df = pd.read_csv(p)
        df.columns = [c.strip() for c in df.columns]
        parts.append(df)
    merged = pd.concat(parts, ignore_index=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], errors="coerce")
    merged = merged.sort_values("timestamp").reset_index(drop=True)

    out = pd.DataFrame({
        "timestamp": merged["timestamp"],
        "Actual Power (W)": merged["actual"],
        "q10 (W)": merged["q10"],
        "q50 (W)": merged["q50"],
        "q90 (W)": merged["q90"],
    })
    out.to_csv(out_path, index=False)
    return out

def load_raw_multivar_5min(path, start_date, src_freq="5min"):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    p_col = _find_col(df, ["power (W)", "power", "load", "active power"])
    t_col = _find_col(df, ["temperature (ºC)", "temperature (C)", "temperature", "temp", "air temperature"])
    v_col = _find_col(df, ["voltage (V)", "voltage"])
    a_col = _find_col(df, ["amperage (A)", "amperage", "current"])
    g_col = _find_col(df, ["generation (w)", "generation", "pv generation"])

    required = {
        "power_w": p_col,
        "temp_c": t_col,
        "voltage_v": v_col,
        "amperage_a": a_col,
        "generation_w": g_col,
    }

    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}. Found {df.columns.tolist()}")

    print(f"[{os.path.basename(path)}] power='{p_col}', temp='{t_col}', voltage='{v_col}', amperage='{a_col}', generation='{g_col}'")

    df["timestamp"] = pd.date_range(start=start_date, periods=len(df), freq=src_freq)

    for src in [p_col, t_col, v_col, a_col, g_col]:
        df[src] = pd.to_numeric(df[src], errors="coerce")

    out = df.rename(columns={
        p_col: "power_w",
        t_col: "temp_c",
        v_col: "voltage_v",
        a_col: "amperage_a",
        g_col: "generation_w",
    })[["timestamp", "power_w", "temp_c", "voltage_v", "amperage_a", "generation_w"]].copy()

    out["hour"] = out["timestamp"].dt.hour
    out["dayofweek"] = out["timestamp"].dt.dayofweek
    out["minute_of_day"] = out["timestamp"].dt.hour * 60 + out["timestamp"].dt.minute
    out["slot_5min"] = out["minute_of_day"] // 5

    return out

def force_5min_grid(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    full_idx = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="5min"
    )

    df = (
        df.set_index("timestamp")
          .reindex(full_idx)
          .rename_axis("timestamp")
          .reset_index()
    )

    num_cols = [c for c in df.columns if c != "timestamp"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[num_cols] = df[num_cols].interpolate(method="linear", limit_direction="both")
    df[num_cols] = df[num_cols].bfill().ffill()

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"] = df["minute_of_day"] // 5

    return df

def add_engineered_features_5min(df):
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    df["lag_5min"] = df["power_w"].shift(1)
    df["lag_10min"] = df["power_w"].shift(2)
    df["lag_1h"] = df["power_w"].shift(12)
    df["lag_24h"] = df["power_w"].shift(288)

    df["ma_1h"] = df["power_w"].shift(1).rolling(window=12, min_periods=1).mean()
    df["ma_3h"] = df["power_w"].shift(1).rolling(window=36, min_periods=1).mean()

    df = df.bfill().ffill()
    return df

def build_tabular_supervised_train(df):
    """
    Rich 5-minute training regime.
    Predict current power from current rich multivariate features.
    """
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    out = df.dropna(subset=TRAIN_FEATURES + ["power_w"]).reset_index(drop=True)
    X = out[TRAIN_FEATURES].to_numpy(dtype=float)
    y = out["power_w"].to_numpy(dtype=float)
    return X, y, TRAIN_FEATURES, out

def aggregate_to_hourly(df_5m):
    """
    Converts 5-minute data to hourly data for deployment-style validation/test.
    Power is averaged within each hour.
    """
    df = df_5m.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["ts_hour"] = df["timestamp"].dt.floor("H")

    ag = (
        df.groupby("ts_hour", as_index=False)
          .agg({
              "power_w": "mean",
              "temp_c": "mean",
              "voltage_v": "mean",
              "amperage_a": "mean",
              "generation_w": "mean",
          })
          .rename(columns={"ts_hour": "timestamp"})
    )

    ag["hour"] = ag["timestamp"].dt.hour
    ag["dayofweek"] = ag["timestamp"].dt.dayofweek
    ag["minute_of_day"] = ag["timestamp"].dt.hour * 60 + ag["timestamp"].dt.minute
    ag["slot_5min"] = ag["minute_of_day"] // 5  # hourly rows will have slot 0,12,...,276

    return ag

def build_hourly_priors_from_train(train_5m):
    """
    Priors learned only from V1 training data.
    Used to fill deployment-unavailable features under strict asymmetry.
    """
    train_h = aggregate_to_hourly(train_5m)

    priors_hour = (
        train_h.groupby("hour")[["voltage_v", "amperage_a", "generation_w"]]
               .median()
               .reset_index()
    )

    priors_global = {
        "voltage_v": float(train_h["voltage_v"].median()),
        "amperage_a": float(train_h["amperage_a"].median()),
        "generation_w": float(train_h["generation_w"].median()),
        "lag_5min": float(train_5m["power_w"].shift(1).median()),
        "lag_10min": float(train_5m["power_w"].shift(2).median()),
        "lag_1h": float(train_5m["power_w"].shift(12).median()),
        "lag_24h": float(train_5m["power_w"].shift(288).median()),
        "ma_1h": float(train_5m["power_w"].shift(1).rolling(12, min_periods=1).mean().median()),
        "ma_3h": float(train_5m["power_w"].shift(1).rolling(36, min_periods=1).mean().median()),
    }

    return priors_hour, priors_global

def make_deployment_style_features(hourly_df, priors_hour, priors_global):
    """
    Deployment-style feature builder:
      known at deployment: temp + time
      unavailable variables: filled from V1 training priors
    Output columns exactly match TRAIN_FEATURES.
    """
    df = hourly_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    df["hour"] = df["timestamp"].dt.hour
    df["dayofweek"] = df["timestamp"].dt.dayofweek
    df["minute_of_day"] = df["timestamp"].dt.hour * 60 + df["timestamp"].dt.minute
    df["slot_5min"] = df["minute_of_day"] // 5

    df = df.merge(priors_hour, on="hour", how="left", suffixes=("", "_prior"))

    for c in ["voltage_v", "amperage_a", "generation_w"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = df[c].where(df[c].notna(), df[c])
        df[c] = df[c].fillna(priors_global[c])

    # overwrite unavailable deployment variables with priors
    df["voltage_v"] = df["voltage_v"].fillna(priors_global["voltage_v"])
    df["amperage_a"] = df["amperage_a"].fillna(priors_global["amperage_a"])
    df["generation_w"] = df["generation_w"].fillna(priors_global["generation_w"])

    # strict asymmetry: no target history available at deployment
    df["lag_5min"] = priors_global["lag_5min"]
    df["lag_10min"] = priors_global["lag_10min"]
    df["lag_1h"] = priors_global["lag_1h"]
    df["lag_24h"] = priors_global["lag_24h"]
    df["ma_1h"] = priors_global["ma_1h"]
    df["ma_3h"] = priors_global["ma_3h"]

    out = df.copy()
    out = out.dropna(subset=["temp_c"]).reset_index(drop=True)

    X = out[TRAIN_FEATURES].to_numpy(dtype=float)
    y = out["power_w"].to_numpy(dtype=float) if "power_w" in out.columns else None
    return X, y, out

def _xgb_supports_quantile_native(dtrain, dval):
    try:
        tmp_params = {
            "objective": "reg:quantileerror",
            "quantile_alpha": 0.5,
            "eta": 0.1,
            "max_depth": 2,
            "tree_method": "hist",
            "seed": SEED,
        }
        _ = xgb.train(
            params=tmp_params,
            dtrain=dtrain,
            num_boost_round=5,
            evals=[(dval, "val")],
            verbose_eval=False
        )
        return True
    except Exception as e:
        print("Quantile objective check failed:", repr(e))
        return False

def train_xgb_point(Xtr, ytr, Xva, yva, feat_cols):
    dtrain = xgb.DMatrix(Xtr, label=ytr, feature_names=feat_cols)
    dval = xgb.DMatrix(Xva, label=yva, feature_names=feat_cols)

    params = dict(
        objective="reg:squarederror",
        eta=0.03,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        **{"lambda": 1.0},
        min_child_weight=1.0,
        tree_method="hist",
        seed=SEED,
    )

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=6000,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=120,
        verbose_eval=False
    )
    return booster, dtrain, dval

def train_xgb_quantile(Xtr, ytr, Xva, yva, feat_cols, alpha):
    dtrain = xgb.DMatrix(Xtr, label=ytr, feature_names=feat_cols)
    dval = xgb.DMatrix(Xva, label=yva, feature_names=feat_cols)

    params = dict(
        objective="reg:quantileerror",
        quantile_alpha=float(alpha),
        eta=0.03,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        **{"lambda": 1.0},
        min_child_weight=1.0,
        tree_method="hist",
        seed=SEED,
    )

    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=7000,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=100,
        verbose_eval=False
    )
    return booster

def build_route1_residual_buckets_from_val_asymmetric(val_df_5m, xgb_point, feat_cols, priors_hour, priors_global):
    """
    Residual buckets are built on V2 validation BUT under deployment-style hourly degraded inference.
    This is the correct asymmetric calibration protocol.
    """
    val_h = aggregate_to_hourly(val_df_5m)
    X_val_dep, y_val_dep, df_val_dep = make_deployment_style_features(val_h, priors_hour, priors_global)

    dm_val = xgb.DMatrix(X_val_dep, feature_names=feat_cols)
    yhat_val = xgb_point.predict(dm_val)

    resid = y_val_dep - yhat_val
    clip_val = np.percentile(np.abs(resid), 99)
    resid = np.clip(resid, -clip_val, clip_val)

    df_tmp = df_val_dep.copy()
    df_tmp["point_pred"] = yhat_val
    df_tmp["residual"] = resid
    df_tmp["hour"] = df_tmp["timestamp"].dt.hour
    df_tmp["dayofweek"] = df_tmp["timestamp"].dt.dayofweek

    res_q = (
        df_tmp.groupby(["hour", "dayofweek"])["residual"]
              .quantile([Q_LO, Q_MID, Q_HI])
              .unstack()
              .rename(columns={Q_LO: "rq10", Q_MID: "rq50", Q_HI: "rq90"})
              .reset_index()
    )

    sigma_global = float(np.std(resid))
    sigma_global = max(sigma_global, 1e-6)

    return res_q, sigma_global, df_tmp

def apply_route1_buckets_hourly(test_timestamps, point_pred, res_q):
    tmp = pd.DataFrame({"timestamp": pd.to_datetime(test_timestamps)})
    tmp["hour"] = tmp["timestamp"].dt.hour
    tmp["dayofweek"] = tmp["timestamp"].dt.dayofweek

    tmp = tmp.merge(res_q, on=["hour", "dayofweek"], how="left")
    tmp[["rq10", "rq50", "rq90"]] = tmp[["rq10", "rq50", "rq90"]].fillna(0.0)

    p10 = point_pred + tmp["rq10"].to_numpy()
    p50 = point_pred + tmp["rq50"].to_numpy()
    p90 = point_pred + tmp["rq90"].to_numpy()

    return enforce_non_crossing(p10, p50, p90)

# =========================
# 5) Load V1 and V2 at 5-minute resolution
# =========================
print("Loading V1 and V2 at 5-minute resolution...")
v1_5m = load_raw_multivar_5min(V1_PATH, V1_START_DATE, SRC_FREQ)
v2_5m = load_raw_multivar_5min(V2_PATH, V2_START_DATE, SRC_FREQ)

full_5m = pd.concat([v1_5m, v2_5m], ignore_index=True)
full_5m = full_5m.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)
full_5m = force_5min_grid(full_5m)
full_5m = add_engineered_features_5min(full_5m)

v1_5m = full_5m[full_5m["timestamp"] < pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)
v2_5m = full_5m[full_5m["timestamp"] >= pd.Timestamp(V2_START_DATE)].copy().reset_index(drop=True)

print("✅ V1 5-min samples:", len(v1_5m))
print("✅ V2 5-min samples:", len(v2_5m))

# =========================
# 6) Split: validation / test
# =========================
test_mask = v2_5m["timestamp"].dt.normalize().isin(TEST_DAYS)

train_df_5m = v1_5m.copy().reset_index(drop=True)
val_df_5m = v2_5m[~test_mask].copy().reset_index(drop=True)
test_df_5m = v2_5m[test_mask].copy().reset_index(drop=True)

print("✅ Train 5-min samples:", len(train_df_5m))
print("✅ Val   5-min samples:", len(val_df_5m))
print("✅ Test  5-min samples:", len(test_df_5m))

# convert test to hourly
test_df_hourly_all = aggregate_to_hourly(test_df_5m)
print("✅ Test hourly samples:", len(test_df_hourly_all))

for d in TEST_DAYS:
    cur = test_df_hourly_all[test_df_hourly_all["timestamp"].dt.normalize() == d].copy()
    if len(cur) != EXPECTED_HOURLY_POINTS_PER_DAY:
        raise ValueError(f"Test day {d.date()} has {len(cur)} hourly points, expected 24.")

# =========================
# 7) Build training priors for missing deployment variables
# =========================
print("\nBuilding V1 priors for deployment-time missing variables...")
priors_hour, priors_global = build_hourly_priors_from_train(train_df_5m)

pd.DataFrame(priors_hour).to_csv(os.path.join(OUT_DIR, "v1_hourly_priors_for_missing_vars.csv"), index=False)
pd.DataFrame([priors_global]).to_csv(os.path.join(OUT_DIR, "v1_global_priors_for_missing_vars.csv"), index=False)

# =========================
# 8) Train Route1 point model on V1 rich 5-minute multivariate
# =========================
print("\nTraining Route1 point model on V1 rich 5-minute multivariate...")
Xtr_all, ytr_all, feat_cols, df_train_sup = build_tabular_supervised_train(train_df_5m)

split_train = int(0.9 * len(df_train_sup))
Xtr, ytr = Xtr_all[:split_train], ytr_all[:split_train]
Xva, yva = Xtr_all[split_train:], ytr_all[split_train:]

t0 = time.time()
xgb_point, dtrain, dval = train_xgb_point(Xtr, ytr, Xva, yva, feat_cols)
print(f"Point model fit time: {time.time()-t0:.2f}s | train_samples={len(ytr_all)} | best_iter={xgb_point.best_iteration}")

# =========================
# 9) Build Route1 residual buckets from V2 validation
#    in deployment-style degraded HOURLY mode
# =========================
print("\nBuilding Route1 residual buckets from V2 validation in deployment-style hourly mode...")
res_q, sigma_global, df_res = build_route1_residual_buckets_from_val_asymmetric(
    val_df_5m=val_df_5m,
    xgb_point=xgb_point,
    feat_cols=feat_cols,
    priors_hour=priors_hour,
    priors_global=priors_global
)

resq_path = os.path.join(OUT_DIR, "xgb_route1_residual_quantiles_from_v2val_asymmetric_hourly.csv")
resid_path = os.path.join(OUT_DIR, "xgb_route1_residuals_from_v2val_asymmetric_hourly.csv")
res_q.to_csv(resq_path, index=False)
df_res.to_csv(resid_path, index=False)

print("✅ Saved residual buckets:", resq_path)
print("✅ Saved residual table  :", resid_path)
print(f"Fallback sigma_global: {sigma_global:.4f}")

# =========================
# 10) Train Route2 quantile models on V1 rich 5-minute multivariate
# =========================
print("\nTraining Route2 probabilistic models on V1 rich 5-minute multivariate...")
HAS_Q = _xgb_supports_quantile_native(dtrain, dval)

if HAS_Q:
    t0 = time.time()
    xgb_q10 = train_xgb_quantile(Xtr, ytr, Xva, yva, feat_cols, Q_LO)
    xgb_q50 = train_xgb_quantile(Xtr, ytr, Xva, yva, feat_cols, Q_MID)
    xgb_q90 = train_xgb_quantile(Xtr, ytr, Xva, yva, feat_cols, Q_HI)
    print(f"Quantile models trained in {time.time()-t0:.2f}s")
else:
    print("⚠️ Native quantile objective unavailable. Route2 will use Normal fallback around point predictions.")

# =========================
# 11) Test on Dec 24 and Dec 30 in degraded hourly mode
# =========================
print("\nTesting on Dec 24 and Dec 30 in degraded hourly mode...")

r1_uq_day_files = []
r2_prob_day_files = []

for k, day_start in enumerate(TEST_DAYS, start=1):
    print("\n" + "=" * 70)
    print(f"Test day {k}: {day_start.date()}")

    day_end = day_start + pd.Timedelta(days=1)

    test_df_h = test_df_hourly_all[
        (test_df_hourly_all["timestamp"] >= day_start) &
        (test_df_hourly_all["timestamp"] < day_end)
    ].copy().reset_index(drop=True)

    if len(test_df_h) != EXPECTED_HOURLY_POINTS_PER_DAY:
        raise ValueError(f"Test day {day_start.date()} does not have 24 hourly samples.")

    X_test_dep, y_true, test_dep_df = make_deployment_style_features(test_df_h, priors_hour, priors_global)
    dm_test = xgb.DMatrix(X_test_dep, feature_names=feat_cols)

    # Route1 point prediction
    point_pred = xgb_point.predict(dm_test)

    # Route1 post-hoc quantiles
    p10_uq, p50_uq, p90_uq = apply_route1_buckets_hourly(test_dep_df["timestamp"].values, point_pred, res_q)

    out_r1 = pd.DataFrame({
        "timestamp": test_dep_df["timestamp"].values,
        "actual": y_true,
        "point_pred": point_pred,
        "q10": p10_uq,
        "q50": p50_uq,
        "q90": p90_uq
    })
    p_r1 = os.path.join(OUT_DIR, f"xgb_route1_uq_asymmetric_hourly_day{k}.csv")
    out_r1.to_csv(p_r1, index=False)
    r1_uq_day_files.append(p_r1)

    # Route2 integrated probabilistic prediction
    if HAS_Q:
        q10_prob = xgb_q10.predict(dm_test)
        q50_prob = xgb_q50.predict(dm_test)
        q90_prob = xgb_q90.predict(dm_test)
        q10_prob, q50_prob, q90_prob = enforce_non_crossing(q10_prob, q50_prob, q90_prob)
    else:
        q50_prob = point_pred.copy()
        q10_prob = point_pred + norm.ppf(0.10) * sigma_global
        q90_prob = point_pred + norm.ppf(0.90) * sigma_global
        q10_prob, q50_prob, q90_prob = enforce_non_crossing(q10_prob, q50_prob, q90_prob)

    out_r2 = pd.DataFrame({
        "timestamp": test_dep_df["timestamp"].values,
        "actual": y_true,
        "q10": q10_prob,
        "q50": q50_prob,
        "q90": q90_prob
    })
    p_r2 = os.path.join(OUT_DIR, f"xgb_route2_prob_asymmetric_hourly_day{k}.csv")
    out_r2.to_csv(p_r2, index=False)
    r2_prob_day_files.append(p_r2)

    print("✅ Saved:")
    print("  Route1_UQ   :", p_r1)
    print("  Route2_Prob :", p_r2)

# =========================
# 12) Merge outputs
# =========================
print("\nMerging outputs...")

merged_route1_path = os.path.join(OUT_DIR, "merged_route1_point_uq_series_xgb_asymmetric_hourly.csv")
merged_route1_df = merge_daily_route1(r1_uq_day_files, merged_route1_path)

merged_route2_path = os.path.join(OUT_DIR, "merged_route2_predictions_prob_xgb_asymmetric_hourly.csv")
merged_route2_df = merge_daily_route2(r2_prob_day_files, merged_route2_path)

print("✅ Merged files written:")
print("  Route1:", merged_route1_path)
print("  Route2:", merged_route2_path)
print("  OUT_DIR:", OUT_DIR)

print("\nRoute1 head:")
print(merged_route1_df.head())

print("\nRoute2 head:")
print(merged_route2_df.head())